In [ ]:
library(tidyr)
library(ggplot2)
library(ggh4x)
library(dplyr)
library(RColorBrewer)
library(tidyselect)
library(tidyr)
library(stringr)
library(ggpubr)
library(labdsv)
library(vegan)
library(readr)
library(ape)
library(lme4)
library(lsmeans)
library(scales)
library(igraph)

In [ ]:
# isolate_cazyme_counts <- read.csv("DOME-Isolates-Subject-Total-CAZyme-Count.csv")
# isolate_cazyme_counts <- read.csv("MSA-DOME-Isolates-Subject-CAZyme-Count.csv")
isolate_cazyme_counts <- read.csv("DOME-Isolates-Subject-Mucin-CAZyme-Count.csv")

In [ ]:
isolate_cazyme_counts <- isolate_cazyme_counts%>%
  rename(Cohort = subject.type, Media = media, CAZyme.Count = cazyme.count)

In [ ]:
concatenated_fastani <- read.csv("DOME-Isolates-fastANI.csv")

In [ ]:
cohort_colors <- c("Cow" = "#990006", "Farmer" = "#386EC2", "Non-Farmer" = "#B5B5B2")

In [ ]:
metadata <- metadata %>%
  dplyr::mutate(Sample.ID = stringr::str_remove(Sample.ID, "^19-"))

In [ ]:
isolate_cazyme_counts <- isolate_cazyme_counts %>%
 left_join(metadata, by = c("Sample.ID", "Cohort"))

In [ ]:
isolate_cazyme_counts <- isolate_cazyme_counts %>%
mutate(Genus = sub(" .*", "", classification))

In [ ]:
isolate_cazyme_counts_with_fastani <- isolate_cazyme_counts %>%
  left_join(concatenated_fastani, by = c("dome.number" = "Isolate.1"))

In [ ]:
# Remove self-comparisons
no_self <- isolate_cazyme_counts_with_fastani %>%
  filter(dome.number != Isolate.2)

# Keep only high ANI pairs (≥ 99.999%)
high_ani_pairs <- no_self %>%
  filter(ANI >= 99.999)

# Initialize list for dereplicated isolates
derep_list <- list()

# Loop over each Sample.ID
for(sample_id in unique(isolate_cazyme_counts_with_fastani$Sample.ID)) {
  
  # Isolates in this sample
  sample_isolates <- isolate_cazyme_counts_with_fastani %>%
    filter(Sample.ID == sample_id) %>%
    pull(dome.number)
  
  # Edges for this sample
  edges <- high_ani_pairs %>%
    filter(Sample.ID == sample_id) %>%
    select(dome.number, Isolate.2) %>%
    distinct()
  
  # Cluster isolates if edges exist
  if(nrow(edges) > 0){
    g <- graph_from_edgelist(as.matrix(edges), directed = FALSE)
    comps <- components(g)
    
    reps <- tibble(
      isolate = names(comps$membership),
      cluster = comps$membership
    ) %>%
      group_by(cluster) %>%
      slice(1) %>%  # pick one representative per cluster
      pull(isolate)
  } else {
    reps <- character(0)
  }
  
  # Add singletons (isolates with no edges)
  singletons <- setdiff(sample_isolates, c(edges$dome.number, edges$Isolate.2))
  
  # Combine cluster reps + singletons
  derep_list[[sample_id]] <- c(reps, singletons)
}

# Combine and ensure Dome.Number is unique
dereplicated_isolates <- isolate_cazyme_counts %>%
  filter(dome.number %in% unlist(derep_list)) %>%
  distinct(dome.number, .keep_all = TRUE)  # ensures one row per Dome.Number

In [ ]:
write.csv(arrange(dereplicated_isolates, dome.number), "DOME-Dereplicated-Isolates-Total-CAZymes.csv", row.names = FALSE)

In [ ]:
write.csv(arrange(dereplicated_isolates, dome.number), "DOME-Dereplicated-Isolates-Mucin-CAZymes.csv", row.names = FALSE)